# Nested Supervised Training

This notebook trains a supervised target on the bundled Breast Cancer Wisconsin dataset. The example reshapes selected columns into a nested `measurements` array so the model sees repeated measurement objects rather than one wide flat row.

The imports mirror the training tutorial. The Breast Cancer records are already buffered as nested JSONL, so the notebook can focus on the schema.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


Each record contains a list of measurement dictionaries plus a diagnosis label. This is intentionally small, but it demonstrates the pattern used for orders with line items, sessions with events, or entities with repeated attributes.


In [2]:
records = pl.read_ndjson("docs/data/breast-cancer.jsonl").head(32)

records.head()


measurements,diagnosis
list[struct[2]],str
"[{""mean_radius"",17.99}, {""mean_texture"",10.38}, … {""mean_smoothness"",0.1184}]","""malignant"""
"[{""mean_radius"",13.54}, {""mean_texture"",14.36}, … {""mean_smoothness"",0.09779}]","""benign"""
"[{""mean_radius"",20.57}, {""mean_texture"",17.77}, … {""mean_smoothness"",0.08474}]","""malignant"""
"[{""mean_radius"",13.08}, {""mean_texture"",15.71}, … {""mean_smoothness"",0.1075}]","""benign"""
"[{""mean_radius"",19.69}, {""mean_texture"",21.25}, … {""mean_smoothness"",0.1096}]","""malignant"""


The nested `Array` defines the measurement context. Inside that array, `name` identifies the measurement and `value` carries the numeric signal. The root-level `diagnosis` field is the supervised target.

In [3]:
model = j2v.Model.from_schema(
    j2v.Array(
        j2v.Category("name", query="[*].measurements[*].name", max_vocab_size=16),
        j2v.Number("value", query="[*].measurements[*].value"),
        name="measurements",
        max_length=5,
    ),
    j2v.Category("diagnosis", query="[*].diagnosis", target=True, max_vocab_size=2),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)


The data module does not need special nested-data code. The schema queries describe where values live, and the data module handles batching and tensorization.

In [4]:
datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

Training here is intentionally minimal: the notebook proves the schema shape and supervised path, not benchmark performance.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The plot is useful for nested schemas because it shows which modules belong to the root record and which belong to the repeated measurement context.

In [6]:
model.plot()

Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=24,935  arrays=2  
fields=3  targets=1  embeds=1  embed=True  n_linear=1
├── measurements [array] 6,608 params
│   record/measurements
│   attention=mha  max_length=5  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
│   ├── name [category] 4,119 params
│   │   record/measurements/name
│   │   query=[*].measurements[*].name  pooling=query  max_vocab_size=16  topk=[]  weight=1.0  n_heads=4  n_linear=1  
│   │   n_bands=8  p_unavailable=0.01
│   └── value [number] 4,007 params
│       record/measurements/value
│       query=[*].measurements[*].value  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  
│       n_bands=8  offset=4
└── diagnosis [category] target 3,593 params
    record/diagnosis
    query=[*].diagnosis  pooling=query  max_vocab_size=2  topk=[]  weight=1.0  n_heads=4  p_prune=1.0  n_linear=1  
    n_bands=8  p_unava

Prediction decodes only configured targets. In this case, the output is the model response for `record/diagnosis`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))

{
│   'record/diagnosis': {
│   │   'state': {
│   │   │   'valued': [0.47202426195144653, 0.47193384170532227, 0.4716031551361084],
│   │   │   'null': [0.25043150782585144, 0.25063249468803406, 0.2506382167339325],
│   │   │   'padded': [0.069636270403862, 0.06965071707963943, 0.06959560513496399],
│   │   │   'masked': [0.1148238331079483, 0.11483940482139587, 0.11491470038890839],
│   │   │   'other': [0.0930841937661171, 0.09294353425502777, 0.09324834495782852]
│   │   },
│   │   'content': {
│   │   │   'value': ['benign', 'benign', 'benign'],
│   │   │   'probability': [0.8512661457061768, 0.8518323302268982, 0.8511031270027161],
│   │   │   'topk': [[], [], []]
│   │   }
│   }
}